# Lección 03 — Patrones de Diseño para Agentes

En este notebook vas a aplicar los 3 patrones de diseño con Claude:

1. **Instrucciones claras** — cómo escribir un system prompt que realmente funcione
2. **Salida estructurada** — cómo hacer que el agente devuelva datos procesables
3. **Responsabilidad única** — cómo dividir tareas complejas entre agentes especializados

In [ ]:
%pip install anthropic python-dotenv -q

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

# Clase Agente de la lección 02 (reutilizable)
class Agente:
    def __init__(self, nombre, instrucciones, herramientas=None, modelo="claude-opus-4-5"):
        self.nombre = nombre
        self.instrucciones = instrucciones
        self.modelo = modelo
        self.herramientas_func = {}
        self.herramientas_schema = []
        self.historial = []
        if herramientas:
            for h in herramientas:
                self.herramientas_schema.append(h["schema"])
                self.herramientas_func[h["schema"]["name"]] = h["funcion"]

    def hablar(self, mensaje, verbose=True):
        self.historial.append({"role": "user", "content": mensaje})
        if verbose:
            print(f"Vos: {mensaje}\n" + "-"*50)
        while True:
            kwargs = dict(
                model=self.modelo, max_tokens=1024,
                system=self.instrucciones, messages=self.historial
            )
            if self.herramientas_schema:
                kwargs["tools"] = self.herramientas_schema
            respuesta = client.messages.create(**kwargs)
            if respuesta.stop_reason == "tool_use":
                uso = next(b for b in respuesta.content if b.type == "tool_use")
                if verbose:
                    print(f"  [{self.nombre} usa: {uso.name}]")
                fn = self.herramientas_func.get(uso.name)
                resultado = fn(**uso.input) if uso.input else fn()
                self.historial.append({"role": "assistant", "content": respuesta.content})
                self.historial.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": uso.id, "content": json.dumps(resultado, ensure_ascii=False)}]})
                continue
            texto = next(b.text for b in respuesta.content if b.type == "text")
            self.historial.append({"role": "assistant", "content": texto})
            if verbose:
                print(f"{self.nombre}: {texto}")
            return texto

print("Setup listo.")

## Patrón 1 — Instrucciones Claras

Vamos a comparar el mismo agente con instrucciones vagas vs. instrucciones claras.
Fijate la diferencia en la calidad de las respuestas.

In [ ]:
# Agente con instrucciones VAGAS
agente_vago = Agente(
    nombre="AgentVago",
    instrucciones="Sos un asistente de viajes. Ayudá al usuario."
)

print("=== INSTRUCCIONES VAGAS ===")
agente_vago.hablar("Quiero ir de vacaciones con 2000 dólares. ¿Qué me recomendás?")

In [ ]:
# Agente con instrucciones CLARAS
agente_claro = Agente(
    nombre="Alex",
    instrucciones="""Sos Alex, un concierge de viajes especializado en destinos latinoamericanos.

Tu rol es:
1. Preguntar por el presupuesto, duración y tipo de experiencia que busca el viajero
2. Recomendar 2-3 destinos específicos con nombre de ciudad y país
3. Para cada destino indicar: precio estimado por semana, mejor época y actividad principal
4. Cerrar siempre con una pregunta para refinar la recomendación

Tono: cálido, entusiasta y profesional. Respondé en español rioplatense.
Límite: máximo 200 palabras por respuesta."""
)

print("\n=== INSTRUCCIONES CLARAS ===")
agente_claro.hablar("Quiero ir de vacaciones con 2000 dólares. ¿Qué me recomendás?")

## Patrón 2 — Salida Estructurada

Ahora hacemos que el agente devuelva un JSON predecible en lugar de texto libre.
Esto es esencial cuando el resultado va a ser procesado por código.

In [ ]:
def obtener_info_destino(destino: str) -> dict:
    """Obtiene información detallada de un destino de viaje."""
    base_datos = {
        "Barcelona": {"disponible": True, "precio_semana_usd": 1800, "mejor_epoca": "mayo-septiembre", "tipo": "ciudad costera"},
        "Cancún": {"disponible": True, "precio_semana_usd": 1100, "mejor_epoca": "noviembre-abril", "tipo": "playa tropical"},
        "Buenos Aires": {"disponible": True, "precio_semana_usd": 900, "mejor_epoca": "marzo-mayo", "tipo": "ciudad cultural"},
        "Tokio": {"disponible": False, "motivo": "cupo lleno para la próxima temporada"},
        "Cartagena": {"disponible": True, "precio_semana_usd": 800, "mejor_epoca": "diciembre-abril", "tipo": "playa colonial"},
    }
    return base_datos.get(destino, {"disponible": False, "motivo": "destino no disponible en nuestro catálogo"})

schema_info_destino = {
    "name": "obtener_info_destino",
    "description": "Obtiene disponibilidad y precio de un destino de viaje específico.",
    "input_schema": {
        "type": "object",
        "properties": {
            "destino": {"type": "string", "description": "Nombre de la ciudad (ej: Barcelona, Cancún)"}
        },
        "required": ["destino"]
    }
}

# Agente con salida estructurada
agente_estructurado = Agente(
    nombre="AnalistadeViajes",
    instrucciones="""Sos un analista de viajes. Cuando el usuario pida recomendaciones:
1. Consultá la disponibilidad de 3 destinos usando la herramienta
2. Devolvé SIEMPRE tu respuesta como un JSON válido con esta estructura exacta:
{
  "recomendaciones": [
    {
      "destino": "nombre",
      "disponible": true/false,
      "precio_semana_usd": numero,
      "mejor_epoca": "meses",
      "tipo": "descripción"
    }
  ],
  "nota_personalizada": "texto con recomendación final"
}
No agregues texto antes ni después del JSON.""",
    herramientas=[{"schema": schema_info_destino, "funcion": obtener_info_destino}]
)

respuesta_json = agente_estructurado.hablar(
    "Quiero opciones de playa con presupuesto de 1500 dólares por semana.",
    verbose=False
)

# Parsear y usar el JSON
try:
    datos = json.loads(respuesta_json)
    print("Respuesta parseada exitosamente:")
    print(f"\nDestinos disponibles:")
    for r in datos["recomendaciones"]:
        if r.get("disponible"):
            print(f"  ✓ {r['destino']} — USD {r.get('precio_semana_usd', 'N/A')}/semana")
    print(f"\nNota: {datos['nota_personalizada']}")
except json.JSONDecodeError:
    print("Respuesta en texto (no JSON):")
    print(respuesta_json)

## Patrón 3 — Responsabilidad Única

Dividimos la tarea de planear un viaje en dos agentes especializados:
- **Agente Destinos** — solo recomienda lugares
- **Agente Logística** — solo planifica el itinerario

Cada uno hace una cosa y la hace bien.

In [ ]:
# Agente 1 — Solo recomienda destinos
agente_destinos = Agente(
    nombre="ExpertoDestinos",
    instrucciones="""Sos un experto en destinos de viaje. Tu único trabajo es:
1. Analizar las preferencias del viajero (clima, presupuesto, tipo de experiencia)
2. Recomendar el destino más adecuado con justificación breve
3. Mencionar por qué ese destino encaja con las preferencias

NO hables de vuelos, hoteles, itinerarios ni logística. Solo el destino.
Sé conciso: máximo 100 palabras.""",
    herramientas=[{"schema": schema_info_destino, "funcion": obtener_info_destino}]
)

# Agente 2 — Solo planifica logística
agente_logistica = Agente(
    nombre="PlanificadorLogistica",
    instrucciones="""Sos un planificador de viajes. Tu único trabajo es:
1. Recibir un destino ya elegido
2. Crear un itinerario día a día para 7 días
3. Sugerir tipo de alojamiento y cómo llegar
4. Estimar cómo distribuir el presupuesto

NO recomiendes destinos alternativos. Solo planificá el viaje al destino dado."""
)

pedido_usuario = "Quiero una semana de playa cálida en Latinoamérica. Presupuesto: USD 1200."

# Paso 1: El agente de destinos elige
print("=== PASO 1: EXPERTO EN DESTINOS ===")
destino_elegido = agente_destinos.hablar(pedido_usuario)

print("\n=== PASO 2: PLANIFICADOR DE LOGÍSTICA ===")
# Paso 2: El agente de logística planifica ese destino
agente_logistica.hablar(f"""El viajero eligió este destino:
{destino_elegido}

Presupuesto total: USD 1200 por 7 días. Planificá el viaje.""")

## Resumen

Aplicaste los 3 patrones de diseño:

| Patrón | Lo que cambiaste | Resultado |
|---|---|---|
| Instrucciones claras | System prompt detallado con rol, pasos y restricciones | Respuestas consistentes y precisas |
| Salida estructurada | Pedir JSON con schema definido | Datos procesables por código |
| Responsabilidad única | Separar en dos agentes especializados | Cada agente hace una cosa y la hace bien |

Estos patrones se pueden combinar: un agente de responsabilidad única con instrucciones claras que devuelve salida estructurada es el diseño más robusto para producción.

---
En la **Lección 04** profundizamos en cómo diseñar herramientas más potentes.